What I have changed from the original implementation:

- Switched dataset from patent phrase matching to Jigsaw Toxic Comment (jigsaw-toxic-comment-classification-challenge).
- Updated data loading to handle Kaggle nested zip structure (train.csv.zip -> train.csv).
- Changed task from single-label classification to true multi-label toxicity prediction (6 labels).
- Replaced score_ascat / score_category label encoding with multi-hot labels vectors:
[toxic, severe_toxic, obscene, threat, insult, identity_hate].
- Kept input as comment_text instead of patent feature concatenation.
- Replaced random/standard stratify split with iterative multi-label stratification (MultilabelStratifiedShuffleSplit), using 90/10 train/val.
- Removed single-class class_weights logic.
- Added per-label imbalance handling with pos_weight = neg_count / pos_count.
- Updated model load for multi-label:
num_labels=6
problem_type="multi_label_classification".
- Kept QLoRA/4-bit + LoRA setup, but adapted training objective for multi-label.
- Updated tokenization pipeline to keep labels directly (no rename_column("score_category", "label")).
- Replaced custom loss from CrossEntropyLoss to BCEWithLogitsLoss(pos_weight=...).
- Replaced compute_metrics from argmax/Pearson to sigmoid-threshold multi-label metrics:
micro_f1, macro_f1, samples_f1, subset_accuracy.
- Updated prediction logic from argmax class output to multi-label sigmoid + threshold output.
- Updated performance reporting to multi-label overall metrics + per-label precision/recall/F1.
- Added/debugged smoke-test flow (small subsets + max_steps) before full run.



# LLAMA3 Fine-tuning for text classification using QLORA

This notebook is an implementaton of:
https://arxiv.org/abs/2305.14314

This notebook is derived from:
https://github.com/adidror005/youtube-videos/blob/main/LLAMA_3_Fine_Tuning_for_Sequence_Classification_Actual_Video.ipynb

Where I adapated the problem and data set from:
https://www.kaggle.com/code/jhoward/getting-started-with-nlp-for-absolute-beginners

### Requirements:
* A GPU with enough memory, 12GB VRAM or more.  Nvidia/Tesla T4 or better would work

### Installs
* They suggest using latest version of transformers
* Must restart after install because the accelerate package used in the hugging face trainer requires it.

### Google Colab
For your convenience open this notebook on google colab with this link
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jkyamog/ml-experiments/blob/main/fine-tuning-qlora/LLAMA_3_Fine_Tuning_for_Sequence_Classification.ipynb)

Add the following secrets to Colab's secret:
* huggingface - hugging face token to access models
* kaggle_user - kaggle username from Setting > Account > API
* kaggle_key - kaggle key from Setting > Account > API

In [ ]:
%pip uninstall -y triton bitsandbytes peft
%pip install -U "numpy<2"
%pip install "torch==2.2.2" "torchvision==0.17.2" tensorboard
%pip install "triton==2.2.0"
%pip install --upgrade \
  "transformers==4.44.2" \
  "datasets==2.18.0" \
  "accelerate==0.33.0" \
  "evaluate==0.4.1" \
  "bitsandbytes==0.43.1" \
  "huggingface_hub==0.24.6" \
  "trl==0.8.6" \
  "peft==0.12.0"
%pip install iterative-stratification




  Using cached torch-2.2.2-cp312-cp312-manylinux1_x86_64.whl.metadata (25 kB)
  Using cached torchvision-0.17.2-cp312-cp312-manylinux1_x86_64.whl.metadata (6.6 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.met

### Big Picture Overview of Parameter Efficient Fine Tuning Methods like LoRA and QLoRA Fine Tuning for Sequence Classification

**The Essence of Fine-tuning**
- LLMs are pre-trained on vast amounts of data for broad language understanding.
- Fine-tuning is crucial for specializing in specific domains or tasks, involving adjustments with smaller, relevant datasets.

**Model Fine-tuning with PEFT: Exploring LoRA and QLoRA**
- Traditional fine-tuning is resource-intensive; PEFT (Parameter Efficient Fine-tuning) makes the process faster and less demanding.
- Focus on two PEFT methods: LoRA and QLoRA.

**The Power of PEFT**
- PEFT modifies only a subset of the LLM's parameters, enhancing speed and reducing memory demands, making it suitable for less powerful devices.

**LoRA: Efficiency through Adapters**
- **Low-Rank Adaptation (LoRA):** Injects small trainable adapters into the pre-trained model.
- **Equation:** For a weight matrix $W$, LoRA approximates $W = W_0 + BA$, where $W_0$ is the original weight matrix, and $BA$ represents the low-rank modification through trainable matrices $B$ and $A$.
- Adapters learn task nuances while keeping the majority of the LLM unchanged, minimizing overhead.

**QLoRA: Compression and Speed**
- **Quantized LoRA (QLoRA):** Extends LoRA by quantizing the model’s weights, further reducing size and enhancing speed.
- **Innovations in QLoRA:**
  1. **4-bit Quantization:** Uses a 4-bit data type, NormalFloat (NF4), for optimal weight quantization, drastically reducing memory usage.
  2. **Low-Rank Adapters:** Fine-tuned with 16-bit precision to effectively capture task-specific nuances.
  3. **Double Quantization:** Reduces quantization constants from 32-bit to 8-bit, saving additional memory without accuracy loss.
  4. **Paged Optimizers:** Manages memory efficiently during training, optimizing for large tasks.

**Why PEFT Matters**
- **Rapid Learning:** Speeds up model adaptation.
- **Smaller Footprint:** Eases deployment with reduced model size.
- **Edge-Friendly:** Fits better on devices with limited resources, enhancing accessibility.

**Conclusion**
- PEFT methods like LoRA and QLoRA revolutionize LLM fine-tuning by focusing on efficiency, facilitating faster adaptability, smaller models, and broader device compatibility.




### Fine-tuning for Text Classification:


#### 1. Text Generation with Classification Label as part of text
- **Approach**: Train the model to generate text that naturally appends the classification label at the end.
- **Input**: "Lorem ipsum dolor sit amet, consectetur adipiscing elit"
- **Output**: "Lorem ipsum dolor sit amet, consectetur adipiscing elit 0.25"
- **Use Case**: This method is useful for classifiying text


#### 2. Sequence Classification Head
- **Approach**: Add a sequence classification head (linear layer) on top of the LLaMa Model transformer. This setup is similar to GPT-2 and focuses on classifying the sentiment based on the last relevant token in the sequence.
    - **Token Positioning**:
        - **With pad_token_id**: The model identifies and ignores padding tokens, using the last non-padding token for classification.
        - **Without pad_token_id**: It defaults to the last token in each sequence.
        - **inputs_embeds**: If embeddings are directly passed (without input_ids), the model cannot identify padding tokens and takes the last embedding in each sequence as the input for classification.
- **Input**: Specific sentences (e.g., "Lorem ipsum dolor sit amet, consectetur adipiscing elit").
- **Output**: Direct classification (e.g., "0.25", "0.50").
- **Training Objective**: Minimize cross-entropy loss between the predicted and the actual sentiment labels.

https://huggingface.co/docs/transformers/main/en/model_doc/llama

### Peft Configs
* Bits and bytes config for quantization
* Lora config for lora

### Going to use Hugginface Transformers trainer class: Main componenents
* Hugging face dataset (for train + eval)
* Data collater
* Compute Metrics
* Class weights since we use custom trainer and also custom weighted loss..
* trainingArgs: like # epochs, learning rate, weight decay etc..



### Login to huggingface hub to put your LLama token so we can access Llama 3 7B Param Pre-trained Model

In [ ]:
import os
from huggingface_hub import login

!rm -f ~/.cache/huggingface/token
!rm -rf ~/.cache/huggingface/hub

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise ValueError("HF_TOKEN env var is not set")

login(token=hf_token)


The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to /root/.cache/huggingface/token
Login successful


###### Imports

In [3]:
import os
import random
import functools
import csv
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import evaluate

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, classification_report, balanced_accuracy_score, accuracy_score

from scipy.stats import pearsonr
from datasets import Dataset, DatasetDict
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)


The easiest way to download Kaggle datasets is to use the Kaggle API. You can install this using pip by running this in a notebook cell:

!pip install kaggle
You need an API key to use the Kaggle API; to get one, click on your profile picture on the Kaggle website, and choose My Account, then click Create New API Token. This will save a file called kaggle.json to your PC. You need to copy this key on your GPU server. To do so, open the file you downloaded, copy the contents, and paste them in the following cell (e.g., creds = '{"username":"xxx","key":"xxx"}'):


In [ ]:
import os
import json

kaggle_username = os.environ.get("KAGGLE_USERNAME")
kaggle_key = os.environ.get("KAGGLE_KEY")

if not kaggle_username or not kaggle_key:
    raise ValueError("KAGGLE_USERNAME or KAGGLE_KEY env var is not set")

creds = json.dumps({"username": kaggle_username, "key": kaggle_key})
creds


Then execute this cell (this only needs to be run once):

In [ ]:
!pip install kaggle

import os
from pathlib import Path

iskaggle = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "")
iskaggle

cred_path = Path("~/.kaggle/kaggle.json").expanduser()
cred_path.parent.mkdir(exist_ok=True)
cred_path.write_text(creds)
cred_path.chmod(0o600)

cred_path


PosixPath('/root/.kaggle/kaggle.json')

Now you can download datasets from Kaggle.

In [6]:
path = Path('jigsaw-toxic-comment-classification-challenge')


if not iskaggle and not path.exists():
    import zipfile,kaggle
    kaggle.api.competition_download_cli(str(path))
    zipfile.ZipFile(f'{path}.zip').extractall(path)

print("path exists:", path.exists())
print("zip exists:", Path(f"{path}.zip").exists())
print("train exists:", (path / "train.csv").exists())
print(list(path.glob("*"))[:20])


100%|██████████| 52.6M/52.6M [00:00<00:00, 211MB/s]



path exists: True
zip exists: True
train exists: False
[PosixPath('jigsaw-toxic-comment-classification-challenge/sample_submission.csv.zip'), PosixPath('jigsaw-toxic-comment-classification-challenge/test.csv.zip'), PosixPath('jigsaw-toxic-comment-classification-challenge/test_labels.csv.zip'), PosixPath('jigsaw-toxic-comment-classification-challenge/train.csv.zip')]


In [7]:
# unzipping the stuff that is still in zip format

import zipfile
from pathlib import Path

path = Path("jigsaw-toxic-comment-classification-challenge")

for z in ["train.csv.zip", "test.csv.zip", "test_labels.csv.zip", "sample_submission.csv.zip"]:
    zp = path / z
    if zp.exists():
        with zipfile.ZipFile(zp, "r") as f:
            f.extractall(path)

print((path / "train.csv").exists())

True


In [8]:
from pathlib import Path
path = Path('jigsaw-toxic-comment-classification-challenge')

!ls {path}

df = pd.read_csv(path/'train.csv')



sample_submission.csv	   test.csv	 test_labels.csv      train.csv
sample_submission.csv.zip  test.csv.zip  test_labels.csv.zip  train.csv.zip


Preparing labels

In [9]:
tox_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

# Ensure numeric 0/1
df[tox_cols] = df[tox_cols].astype("float32")

# Model input text column
df["input"] = df["comment_text"]

# Multi-label target vector
df["labels"] = df[tox_cols].values.tolist()

df = df[["input", "labels"]]
df.head()

# transforms dataset from its original structure to a simple two column structure
# where the input is the comment_text
# and the labels are multi-label target vectors like [1, 1, 0, 0, 1, 0] where
# each number corresponds to if a category is considered true or not


,input,labels
0,Explanation\nWhy the edits made under my usern...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
1,D'aww! He matches this background colour I'm s...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
2,"Hey man, I'm really not trying to edit war. It...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
3,"""\nMore\nI can't make any real suggestions on ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"
4,"You, sir, are my hero. Any chance you remember...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]"


### Convert from Pandas DataFrame to Hugging Face Dataset
* Also let's shuffle the training set.
* We put the components train,val,test into a DatasetDict so we can access them later with HF trainer.
* Later we will add a tokenized dataset


In [10]:

# we are doing a split with multi-label stratification to ensure label-balance, prevent
# unstable/biased data

import numpy as np
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from datasets import Dataset, DatasetDict

tox_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

# df is expected to already contain:
# - input (text)
# - labels (list of 6 floats/ints)
assert "input" in df.columns and "labels" in df.columns

# Build Y matrix for stratification
Y = np.array(df["labels"].tolist(), dtype=np.int64)

# 90/10 split with multi-label stratification
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_idx, val_idx = next(msss.split(df, Y))

df_train = df.iloc[train_idx].reset_index(drop=True)
df_val = df.iloc[val_idx].reset_index(drop=True)

# Safety checks
required_cols = ["input", "labels"]
for name, part in [("train", df_train), ("val", df_val)]:
    missing = [c for c in required_cols if c not in part.columns]
    assert not missing, f"{name} missing columns: {missing}"
    assert part["input"].notna().all(), f"{name} has null input"
    assert part["labels"].notna().all(), f"{name} has null labels"

# Convert to HF datasets
dataset_train = Dataset.from_pandas(df_train, preserve_index=False)
dataset_val = Dataset.from_pandas(df_val, preserve_index=False)

dataset = DatasetDict({
    "train": dataset_train,
    "val": dataset_val,
})

# Optional quick prevalence check
def prevalence(frame):
    y = np.array(frame["labels"].tolist(), dtype=np.float32)
    return pd.Series(y.mean(axis=0), index=tox_cols)

print("Train prevalence:\n", prevalence(df_train).round(4))
print("\nVal prevalence:\n", prevalence(df_val).round(4))
print("\nSizes:", len(df_train), len(df_val))


Train prevalence:
 toxic            0.0958
severe_toxic     0.0100
obscene          0.0529
threat           0.0030
insult           0.0494
identity_hate    0.0088
dtype: float32

Val prevalence:
 toxic            0.0958
severe_toxic     0.0100
obscene          0.0530
threat           0.0030
insult           0.0494
identity_hate    0.0088
dtype: float32

Sizes: 143613 15958


Doing pos weight balancing

In [12]:
tox_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

# df_train["labels"] is expected to be list-like length 6
Y_train = np.array(df_train["labels"].tolist(), dtype=np.float32)  # shape [N, 6]

pos_counts = Y_train.sum(axis=0)                  # positives per label
neg_counts = Y_train.shape[0] - pos_counts        # negatives per label

# BCEWithLogitsLoss uses: pos_weight[c] > 1 to upweight rare positives
pos_weight = neg_counts / np.clip(pos_counts, 1.0, None)
pos_weight = torch.tensor(pos_weight, dtype=torch.float32)

print("Positive counts per label:")
print(dict(zip(tox_cols, pos_counts.astype(int))))

print("\nComputed pos_weight per label:")
print(dict(zip(tox_cols, pos_weight.tolist())))

# labels with proportionally low pos_counts are weighted higher

pos_weight

Positive counts per label:
{'toxic': 13765, 'severe_toxic': 1435, 'obscene': 7604, 'threat': 430, 'insult': 7089, 'identity_hate': 1264}

Computed pos_weight per label:
{'toxic': 9.433199882507324, 'severe_toxic': 99.07874298095703, 'obscene': 17.886507034301758, 'threat': 332.9837341308594, 'insult': 19.258569717407227, 'identity_hate': 112.61788177490234}


tensor([  9.4332,  99.0787,  17.8865, 332.9837,  19.2586, 112.6179])

## Load LLama model with 4 bit quantization as specified in bits and bytes and prepare model for peft training

### Model Name

In [13]:
model_name = "meta-llama/Llama-3.1-8B"

#### Quantization Config (for QLORA)

In [14]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit = True, # enable 4-bit quantization
    bnb_4bit_quant_type = 'nf4', # information theoretically optimal dtype for normally distributed weights
    bnb_4bit_use_double_quant = True, # quantize quantized weights //insert xzibit meme
    bnb_4bit_compute_dtype = torch.bfloat16 # optimized fp format for ML
)


#### Lora Config

In [15]:
lora_config = LoraConfig(
    r = 16, # the dimension of the low-rank matrices
    lora_alpha = 8, # scaling factor for LoRA activations vs pre-trained weight activations
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout = 0.05, # dropout probability of the LoRA layers
    bias = 'none', # wether to train bias weights, set to 'none' for attention layers
    task_type = 'SEQ_CLS'
)

#### Load model
* AutomodelForSequenceClassification
* Num Labels is # of classes


In [ ]:
import os

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    num_labels=len(tox_cols),
    problem_type="multi_label_classification",
    token=os.environ.get("HF_TOKEN"),
    # adding problem_type to be explict, ensure safety
)

model


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

`low_cpu_mem_usage` was None, now set to True since model is quantized.


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Llama-3.1-8B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


LlamaForSequenceClassification(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNor

* prepare_model_for_kbit_training() function to preprocess the quantized model for training.

In [17]:
model = prepare_model_for_kbit_training(model)
model

LlamaForSequenceClassification(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNor

* get_peft_model prepares a model for training with a PEFT method such as LoRA by wrapping the base model and PEFT configuration with get_peft_model

In [18]:
model = get_peft_model(model, lora_config)
model

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): LlamaForSequenceClassification(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaSdpaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
           

### Load the tokenizer

#### Since LLAMA3 pre-training doesn't have EOS token
* Set the pad_token_id to eos_token_id
* Set pad token ot eos_token

In [19]:
tokenizer = AutoTokenizer.from_pretrained(model_name, add_prefix_space=True)

tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

#### Update some model configs
* Must use .cache = False as below or it crashes from my experience

In [20]:
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model.config.pretraining_tp = 1

# Trainer Components
* model
* tokenizer
* training arguments
* train dataset
* eval dataset
* Data Collater
* Compute Metrics
* class_weights: In our case since we are using a custom trainer so we can use a weighted loss we will subclass trainer and define the custom loss.

#### Create LLAMA tokenized dataset which will house our train/val parts during the training process but after applying tokenization

In [ ]:
# MAX_LEN = 128
MAX_LEN = 512
# change back to 512 when going to nautilus

def llama_preprocessing_function(examples):
    return tokenizer(examples["input"], truncation=True, max_length=MAX_LEN)

tokenized_datasets = dataset.map(llama_preprocessing_function, batched=True)

# keep multi-label targets as "labels" (already present in your df)
tokenized_datasets.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

# going from raw text data in the 'input' column to token IDs and attention masks that the model can understand, and then converting those into PyTorch tensors for efficient training.

# converting to tensors enables the core ML functionality to efficiently compute gradients and perform backpropagation during training, as well as to leverage GPU acceleration if available.
# a standardized process, doesn't matter what model or task you're working on, you typically want to convert your data into tensors before feeding it into the model for training or inference, because that's the format that deep learning frameworks like PyTorch and TensorFlow are optimized for.

# this section basically takes each dataset row, and tokenizes the 'input' text column into input IDs and attention masks, which are the formats the model expects. It also removes some unnecessary columns and renames the label column to 'label' for compatibility with the Trainer. Finally, it sets the format to PyTorch tensors for efficient training.
# the tokenizer isn't generating new token ids here, it is just obtaining them since the model has already been pretrained and the tokenizer is just converting the raw text into the token IDs that correspond to the model's vocabulary.
# preparing the dataset for training in a similar manner that you would for pre-training, except here the objective is supervised classification rather than next-token prediction, and we are configuring the model in such a way that during training it will only be making tweaks to the lora layers and the classification head, while the rest of the model's pretrained weights remain mostly frozen.

Map:   0%|          | 0/143613 [00:00<?, ? examples/s]

Map:   0%|          | 0/15958 [00:00<?, ? examples/s]

## Data Collator
A **data collator** prepares batches of data for training or inference in machine learning, ensuring uniform formatting and adherence to model input requirements. This is especially crucial for variable-sized inputs like text sequences.

### Functions of Data Collator

1. **Padding:** Uniformly pads sequences to the length of the longest sequence using a special token, allowing simultaneous batch processing.
2. **Batching:** Groups individual data points into batches for efficient processing.
3. **Handling Special Tokens:** Adds necessary special tokens to sequences.
4. **Converting to Tensor:** Transforms data into tensors, the required format for machine learning frameworks.

### `DataCollatorWithPadding`

The `DataCollatorWithPadding` specifically manages padding, using a tokenizer to ensure that all sequences are padded to the same length for consistent model input.

- **Syntax:** `collate_fn = DataCollatorWithPadding(tokenizer=tokenizer)`
- **Purpose:** Automatically pads text data to the longest sequence in a batch, crucial for models like BERT or GPT.
- **Tokenizer:** Uses the provided `tokenizer` for sequence processing, respecting model-specific vocabulary and formatting rules.

This collator is commonly used with libraries like Hugging Face's Transformers, facilitating data preprocessing for various NLP models.


In [32]:
collate_fn = DataCollatorWithPadding(tokenizer=tokenizer)

# takes the tokenized examples (of varying length) and at batch time pads them to the longest sequence in the batch, knows what id to insert for padding and how to update the attention mask accordingly, so that the model doesn't attend to the padding tokens. Once this is done we have uniform length tensors per batch
# batch time means when a group of examples are chosen to go through the model, before they do go through it they are padded to the same length so they can be processed together in a batch, this is more efficient than processing each example one at a time.

# define which metrics to compute for evaluation


In [33]:
# doing different compute_metrics bc we are doing multi-label, also pearson wouldn't have been good even if it was exclusively categorization

from sklearn.metrics import f1_score, accuracy_score
import numpy as np
import torch

def compute_metrics(eval_pred):
    logits, labels = eval_pred  # logits: [N, 6], labels: [N, 6]

    probs = 1 / (1 + np.exp(-logits))      # sigmoid
    preds = (probs >= 0.5).astype(int)     # threshold

    return {
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        #pools label decisions across all samples
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        # calculates the F1 score for each label independently and then averages them, treating all labels equally regardless of their frequency in the dataset. This is useful when you want to evaluate the performance on each label separately and give equal importance to all labels, even if some are rare.
        "samples_f1": f1_score(labels, preds, average="samples", zero_division=0),
        # how accurate is each example
        "subset_accuracy": accuracy_score(labels, preds),  # exact-match accuracy
        # how many examples got all labels correct
    }

### Define custom trainer with pos weight
* We will have a custom loss function that deals with the pos weight and have pos weight as additional argument in constructor

Use this one, uses pos_weight rather than class_weight bc doing multi-label classification, also uses BCE instead of cross entropy

In [34]:
class CustomTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        if pos_weight is not None:
            # keep as float tensor; move to model device at loss time
            self.pos_weight = pos_weight.float()
        else:
            self.pos_weight = None

    def compute_loss(self, model, inputs, return_outputs=False):

        # extract labels and convert to float for BCEWithLogitsLoss, which expects float targets
        labels = inputs.pop("labels").float()   # [batch, num_labels]

        # forward pass through the model to get logits
        outputs = model(**inputs)

        # Extract logits assuming they are directly outputted by the model
        logits = outputs.get("logits")          # [batch, num_labels]

        if self.pos_weight is not None:
            loss_fn = torch.nn.BCEWithLogitsLoss(
                pos_weight=self.pos_weight.to(logits.device)
            )
        else:
            loss_fn = torch.nn.BCEWithLogitsLoss()

        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

    # in BCE, essentially treating each label as a separate binary classification problem, and the pos_weight allows us to give more importance to the positive examples for each label, which can help the model learn better in cases of class imbalance. The compute_loss function is overridden to implement this custom loss calculation during training.
    # binary classify y to {0, 1}, if y = 1 then do -log(p) where p is the sigmoid prob, if y = 0 then -log(1-p), then avergae that across all labels
    # when u apply pos_weight, only modify the positive term, so if y = 1 then do -pos_weight*log(p) which gives more penalty to getting positive examples wrong, which can help the model learn better in cases where the positive class is underrepresented in the dataset, negative term remains unchanged
    # bce has equal penalty for missing positive and negative labels when pos_weight is not used,
    # applying pos_weight amplifies positive miss errors even more

# define training args

In [35]:
training_args = TrainingArguments(
    output_dir = 'sequence_classification',
    learning_rate = 1e-4,
    # learning rate here pretty much represents how much the model's weights (specifically the adapter weights of the lora matrices) are updated during training in response to the computed loss, it is different from lora_alpha which represents how strong the adapter signal is rahter than how much the weights are updated in response to that signal
    per_device_train_batch_size = 8,
    # training happens in batches of size 8, which means that the model will process 8 examples at a time before updating the weights. This is a common practice to balance memory usage and training speed, as larger batch sizes can lead to faster training but require more memory, while smaller batch sizes are more memory efficient but can lead to noisier gradient estimates and slower training.
    per_device_eval_batch_size = 8,
    # when evaluating (which happens once at the end of each epoch in this case), examples will be evaluated in batches of 8, which can speed up the evaluation process while still fitting within memory constraints. The same applies to training, where the model will process 8 examples at a time before updating the weights.
    num_train_epochs = 2,
    weight_decay = 0.01,
    # discourages large parameter values by directly bringing down the value of the weight by a certain amount during each optimizer step, which can help prevent overfitting and improve generalization to unseen data.
    eval_strategy = 'epoch',
    save_strategy = 'epoch',
    # running evaluation and saving checkpoints at the end of each epoch, which is a common practice to monitor training progress and keep track of the best model based on evaluation metrics.
    load_best_model_at_end = True,

    metric_for_best_model="eval_macro_f1",
    # tells trainer that metric to base off of is eval macro f1
    # chose this bc task is imbalanced multi-label so we care about rare labels
    # w this sparse labels matter
    greater_is_better=True,
    # confirms that higher metric values are better, which if we are using f1 then this is true
    logging_steps=50,
    # logs training stats every 50 optimizer steps to monitor progress
    report_to="none",
    # no external logger integrations
)

# high level picture: batches respectively go through the trainer model and when a batch goes thru the weights are updated based on the computed loss given its predictions, and then once all of the batches in the epoch have gone through, evaluation happens based on the most recent model weights and it is done on a held-out data (not the same data that the model was trained on),

# Since you’re using custom metrics, set metric_for_best_model (for example balanced_accuracy or macro_f1) so “best model” is chosen by the metric you care about.

#### Define custom trainer

In [36]:
trainer = CustomTrainer(
    model = model,
    args = training_args,
    train_dataset = tokenized_datasets['train'],
    eval_dataset = tokenized_datasets['val'],
    tokenizer = tokenizer,
    data_collator = collate_fn,
    compute_metrics = compute_metrics,
    pos_weight=pos_weight,
    # using pos_weight instead of class weight
)

# puts all the pieces we have been defining together into the Trainer, which will handle the training loop, evaluation,
# and everything in between. We pass in our model, training arguments, datasets, tokenizer, data collator, custom compute_metrics
# function, and class weights for loss computation. Once this Trainer is set up, we can call trainer.train() to start the training
# process, and it will use all the configurations we've defined to train the model and evaluate it at the end of each epoch.

* https://huggingface.co/docs/transformers/en/training

Smoke Test Before Going Over to Nautilus

In [ ]:
# # 1) tiny debug subsets
# debug_train = tokenized_datasets["train"].select(range(1000))
# debug_val = tokenized_datasets["val"].select(range(200))

# # 2) debug training args
# debug_args = TrainingArguments(
#     output_dir="sequence_classification_debug",
#     learning_rate=1e-4,
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,
#     num_train_epochs=1,
#     max_steps=20,                 # smoke test cap
#     weight_decay=0.01,
#     eval_strategy="steps",
#     eval_steps=10,
#     save_strategy="steps",
#     save_steps=20,
#     load_best_model_at_end=False,  # keep simple for smoke test
#     logging_steps=10,
#     report_to="none",
# )

# # 3) debug trainer
# debug_trainer = CustomTrainer(
#     model=model,
#     args=debug_args,
#     train_dataset=debug_train,
#     eval_dataset=debug_val,
#     tokenizer=tokenizer,
#     data_collator=collate_fn,
#     compute_metrics=compute_metrics,
#     pos_weight=pos_weight,
# )

# # 4) run smoke train
# debug_result = debug_trainer.train()
# print(debug_result.metrics)

# # 5) quick eval
# debug_eval = debug_trainer.evaluate()
# print(debug_eval)


max_steps is given, it will override any value given in num_train_epochs
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:460: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss,Validation Loss,Micro F1,Macro F1,Samples F1,Subset Accuracy
10,3.193500,2.879459,0.085366,0.055490,0.018833,0.615000
20,2.633800,2.799332,0.101266,0.066972,0.023167,0.620000


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:460: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


{'train_runtime': 443.201, 'train_samples_per_second': 0.361, 'train_steps_per_second': 0.045, 'total_flos': 824086787850240.0, 'train_loss': 2.913656711578369, 'epoch': 0.16}


{'eval_loss': 2.7993323802948, 'eval_micro_f1': 0.10126582278481013, 'eval_macro_f1': 0.06697191697191697, 'eval_samples_f1': 0.023166666666666665, 'eval_subset_accuracy': 0.62, 'eval_runtime': 98.4306, 'eval_samples_per_second': 2.032, 'eval_steps_per_second': 0.254, 'epoch': 0.16}


### Run trainer!

In [29]:
train_result = trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:460: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

#### Let's check the results
* I wrapped in a function a convenient way add the predictions

In [38]:
def make_predictions(model, df, threshold=0.5, batch_size=32):

    # put model in eval mode to disable dropout and other training-specific behavior, ensuring that the model's predictions are deterministic and consistent during inference, which is important for getting reliable predictions on new data.
    model.eval()

    # use the same devices as the model parameters, avoid device mismatch errors
    device = next(model.parameters()).device

    # getting the list of sentences back from the dataframe
    sentences = df["input"].tolist()

    # get logits from each mini-batch and store them in a list
    all_logits = []

    # batch inference loop, go through all the sentences in batches
    for i in range(0, len(sentences), batch_size):
        # batch of sentences to process together, this is more efficient than processing each sentence one at a time, especially when using a GPU, as it allows for parallel computation and better utilization of resources.
        batch_sentences = sentences[i:i + batch_size]

        # tokenize batch to what the model expects, and move tensors to the same device as the model to avoid device mismatch errors
        inputs = tokenizer(
            batch_sentences,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LEN
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # forward pass without gradient calculation since we are just doing inference
        with torch.no_grad():
            # forward pass through the model to get logits, which are the raw outputs before applying sigmoid for multi-label classification
            outputs = model(**inputs)

            # add the logits for this batch to our list, detaching from the computation graph (internal record of operations for backpropagation) and moving to CPU to avoid memory issues, let GPU memory focus on forward passes
            all_logits.append(outputs.logits.detach().cpu())

    # stacks tensors from all batches into a single tensor
    logits = torch.cat(all_logits, dim=0)                      # [N, 6]
    # applies sigmoid to convert logits to probabilities
    probs = torch.sigmoid(logits).numpy()                      # [N, 6]
    # applies the threshold to get binary predictions for each label
    preds = (probs >= threshold).astype(int)                   # [N, 6]

    tox_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

    # return copy of original dataframe with new columns for predicted probabilities and labels, as well as convenience columns for each toxicity type, don't want to modify original dataframe in case we want to compare predictions to true labels or do other analyses, so we create a copy and add the new columns there.

    df = df.copy()
    df["pred_probs"] = probs.tolist()
    df["pred_labels"] = preds.tolist()

    # Optional convenience columns per label
    for j, col in enumerate(tox_cols):
        df[f"pred_{col}"] = preds[:, j]
        df[f"prob_{col}"] = probs[:, j]

    return df


### Analyze performance

In [39]:
from sklearn.metrics import f1_score, accuracy_score, classification_report
import numpy as np
import pandas as pd

def get_performance_metrics(df_pred):
    """
    Expects dataframe with:
      - true labels in column: 'labels' (list of 0/1 length 6)
      - predicted labels in column: 'pred_labels' (list of 0/1 length 6)
    """

    tox_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

    # Convert list-columns to 2D numpy arrays: shape [N, 6]
    y_true = np.array(df_pred["labels"].tolist(), dtype=int)
    y_pred = np.array(df_pred["pred_labels"].tolist(), dtype=int)

    # Global multi-label metrics
    metrics = {
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "samples_f1": f1_score(y_true, y_pred, average="samples", zero_division=0),
        "subset_accuracy": accuracy_score(y_true, y_pred),  # exact match of all 6 labels
    }

    print("Overall metrics:")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

    # Per-label summary table - one row per toxicity label, with
    # performance for that label treated as its own binary task
    per_label_rows = []
    for j, col in enumerate(tox_cols):
        report_j = classification_report(
            y_true[:, j],
            y_pred[:, j],
            output_dict=True,
            zero_division=0
        )
        per_label_rows.append({
            "label": col,
            "precision": report_j["1"]["precision"],
            "recall": report_j["1"]["recall"],
            "f1": report_j["1"]["f1-score"],
            "support_pos": int(report_j["1"]["support"]),
        })

    per_label_df = pd.DataFrame(per_label_rows).sort_values("label")
    print("\nPer-label metrics (positive class):")
    print(per_label_df.to_string(index=False))

    return metrics, per_label_df


In [ ]:
# # DEBUG VERSION (fast smoke test)
# # Switch back later: use full df_val instead of .head(500)
# df_val_pred = make_predictions(model, df_val.head(500), threshold=0.5, batch_size=32)

# # Evaluate predictions on that same debug dataframe
# metrics, per_label_df = get_performance_metrics(df_val_pred)

# # Inspect prediction dataframe
# df_val_pred.head()


# FULL VERSION
df_val_pred = make_predictions(model, df_val, threshold=0.5, batch_size=32)
metrics, per_label_df = get_performance_metrics(df_val_pred)
df_val_pred.head()


Overall metrics:
micro_f1: 0.1047
macro_f1: 0.0710
samples_f1: 0.0183
subset_accuracy: 0.6300

Per-label metrics (positive class):
        label  precision   recall       f1  support_pos
identity_hate   0.000000 0.000000 0.000000            2
       insult   0.092593 0.161290 0.117647           31
      obscene   0.175000 0.318182 0.225806           22
 severe_toxic   0.000000 0.000000 0.000000            2
       threat   0.000000 0.000000 0.000000            0
        toxic   0.058824 0.139535 0.082759           43


,input,labels,pred_probs,pred_labels,pred_toxic,prob_toxic,pred_severe_toxic,prob_severe_toxic,pred_obscene,prob_obscene,pred_threat,prob_threat,pred_insult,prob_insult,pred_identity_hate,prob_identity_hate
0,"""\n\nCongratulations from me as well, use the ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[0.10431075841188431, 0.019973332062363625, 0....","[0, 0, 0, 0, 0, 0]",0,0.104311,0,0.019973,0,0.017029,0,0.000896,0,0.025651,0,0.001431
1,There's no need to apologize. A Wikipedia arti...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[0.06298104673624039, 0.00261631584726274, 0.0...","[0, 0, 0, 0, 0, 0]",0,0.062981,0,0.002616,0,0.001024,0,0.000109,0,0.033751,0,0.000406
2,"""\nOk. But it will take a bit of work but I ca...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[0.0917477235198021, 0.07455866783857346, 0.06...","[0, 0, 0, 0, 0, 0]",0,0.091748,0,0.074559,0,0.060495,0,0.009531,0,0.022796,0,0.002040
3,"""\n\n """"Mainland Asia"""" includes """"the lower b...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[0.9952817559242249, 0.018792591989040375, 0.9...","[1, 0, 1, 0, 1, 1]",1,0.995282,0,0.018793,1,0.964580,0,0.010956,1,0.994746,1,0.831640
4,pretty much everyone from warren county/surrou...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]","[0.060053616762161255, 0.0051911999471485615, ...","[0, 0, 0, 0, 0, 0]",0,0.060054,0,0.005191,0,0.000163,0,0.000353,0,0.002674,0,0.000687


### Saving the model trainer state and model adapters

In [ ]:
# # DEBUG VERSION
# metrics = debug_result.metrics
# max_train_samples = len(debug_train)
# metrics["train_samples"] = min(max_train_samples, len(debug_train))

# debug_trainer.log_metrics("train", metrics)
# debug_trainer.save_metrics("train", metrics)
# debug_trainer.save_state()

# FULL VERSION
metrics = train_result.metrics
max_train_samples = len(dataset_train)
metrics["train_samples"] = min(max_train_samples, len(dataset_train))

trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()


***** train metrics *****
  epoch                    =       0.16
  total_flos               =   767490GF
  train_loss               =     2.9137
  train_runtime            = 0:07:23.20
  train_samples            =       1000
  train_samples_per_second =      0.361
  train_steps_per_second   =      0.045


#### Saving the adapter model
* Note this doesn't save the entire model. It only saves the adapters.

In [ ]:
# # DEBUG VERSION
# debug_trainer.save_model("saved_model_debug")

# FULL VERSION
trainer.save_model("saved_model")

### Save to google drive
Make sure before disconnecting and deleting the Colab runtime you save your model for future inference use

In [ ]:
!cp -r sequence_classification /content/drive/MyDrive/Colab

In [ ]:
!cp -r saved_model /content/drive/MyDrive/Colab